In [1]:
# ==========================================================
# Notebook 07: Cross-Encoder Re-ranking
# ==========================================================

import pandas as pd
import numpy as np

from sentence_transformers import CrossEncoder

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
retrieved_candidates = pd.DataFrame(
    {
        "candidate": [
            "candidate_1.pdf",
            "candidate_2.pdf",
            "candidate_3.pdf",
            "candidate_4.pdf",
            "candidate_5.pdf",
        ],
        "resume_text": [
            "Python SQL Machine Learning Deep Learning TensorFlow",
            "Python LangChain RAG LlamaIndex FAISS NLP",
            "Java Spring Boot Microservices Docker",
            "Python NLP Transformers PyTorch HuggingFace",
            "Data Analysis SQL Excel Tableau",
        ],
        "faiss_score": [0.93, 0.91, 0.88, 0.86, 0.81],
    }
)

retrieved_candidates

,candidate,resume_text,faiss_score
0,candidate_1.pdf,Python SQL Machine Learning Deep Learning Tens...,0.93
1,candidate_2.pdf,Python LangChain RAG LlamaIndex FAISS NLP,0.91
2,candidate_3.pdf,Java Spring Boot Microservices Docker,0.88
3,candidate_4.pdf,Python NLP Transformers PyTorch HuggingFace,0.86
4,candidate_5.pdf,Data Analysis SQL Excel Tableau,0.81


In [3]:
job_description = """
Machine Learning Engineer

Required Skills:
- Python
- NLP
- Transformers
- LangChain
- RAG
- Vector Databases
- HuggingFace
"""

In [4]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder loaded.")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Cross-Encoder loaded.


In [5]:
pairs = []

for _, row in retrieved_candidates.iterrows():

    pairs.append((job_description, row["resume_text"]))

pairs

[('\nMachine Learning Engineer\n\nRequired Skills:\n- Python\n- NLP\n- Transformers\n- LangChain\n- RAG\n- Vector Databases\n- HuggingFace\n',
  'Python SQL Machine Learning Deep Learning TensorFlow'),
 ('\nMachine Learning Engineer\n\nRequired Skills:\n- Python\n- NLP\n- Transformers\n- LangChain\n- RAG\n- Vector Databases\n- HuggingFace\n',
  'Python LangChain RAG LlamaIndex FAISS NLP'),
 ('\nMachine Learning Engineer\n\nRequired Skills:\n- Python\n- NLP\n- Transformers\n- LangChain\n- RAG\n- Vector Databases\n- HuggingFace\n',
  'Java Spring Boot Microservices Docker'),
 ('\nMachine Learning Engineer\n\nRequired Skills:\n- Python\n- NLP\n- Transformers\n- LangChain\n- RAG\n- Vector Databases\n- HuggingFace\n',
  'Python NLP Transformers PyTorch HuggingFace'),
 ('\nMachine Learning Engineer\n\nRequired Skills:\n- Python\n- NLP\n- Transformers\n- LangChain\n- RAG\n- Vector Databases\n- HuggingFace\n',
  'Data Analysis SQL Excel Tableau')]

In [6]:
cross_scores = cross_encoder.predict(pairs)

cross_scores

array([ -4.400734 ,  -1.9629738, -11.106098 ,  -0.7634839, -10.903401 ],
      dtype=float32)

In [7]:
retrieved_candidates["cross_encoder_score"] = np.round(cross_scores, 3)

retrieved_candidates

,candidate,resume_text,faiss_score,cross_encoder_score
0,candidate_1.pdf,Python SQL Machine Learning Deep Learning Tens...,0.93,-4.401
1,candidate_2.pdf,Python LangChain RAG LlamaIndex FAISS NLP,0.91,-1.963
2,candidate_3.pdf,Java Spring Boot Microservices Docker,0.88,-11.106
3,candidate_4.pdf,Python NLP Transformers PyTorch HuggingFace,0.86,-0.763
4,candidate_5.pdf,Data Analysis SQL Excel Tableau,0.81,-10.903


In [8]:
reranked_df = retrieved_candidates.sort_values(
    by="cross_encoder_score", ascending=False
).reset_index(drop=True)

reranked_df.index += 1

reranked_df

,candidate,resume_text,faiss_score,cross_encoder_score
1,candidate_4.pdf,Python NLP Transformers PyTorch HuggingFace,0.86,-0.763
2,candidate_2.pdf,Python LangChain RAG LlamaIndex FAISS NLP,0.91,-1.963
3,candidate_1.pdf,Python SQL Machine Learning Deep Learning Tens...,0.93,-4.401
4,candidate_5.pdf,Data Analysis SQL Excel Tableau,0.81,-10.903
5,candidate_3.pdf,Java Spring Boot Microservices Docker,0.88,-11.106


In [9]:
comparison_df = reranked_df[["candidate", "faiss_score", "cross_encoder_score"]]

comparison_df

,candidate,faiss_score,cross_encoder_score
1,candidate_4.pdf,0.86,-0.763
2,candidate_2.pdf,0.91,-1.963
3,candidate_1.pdf,0.93,-4.401
4,candidate_5.pdf,0.81,-10.903
5,candidate_3.pdf,0.88,-11.106


In [10]:
def rerank_candidates(job_description, candidate_df, cross_encoder):

    pairs = []

    for _, row in candidate_df.iterrows():

        pairs.append((job_description, row["resume_text"]))

    scores = cross_encoder.predict(pairs)

    result_df = candidate_df.copy()

    result_df["cross_encoder_score"] = scores

    result_df = result_df.sort_values(by="cross_encoder_score", ascending=False)

    result_df = result_df.reset_index(drop=True)

    return result_df

In [11]:
final_ranking = rerank_candidates(job_description, retrieved_candidates, cross_encoder)

final_ranking

,candidate,resume_text,faiss_score,cross_encoder_score
0,candidate_4.pdf,Python NLP Transformers PyTorch HuggingFace,0.86,-0.763484
1,candidate_2.pdf,Python LangChain RAG LlamaIndex FAISS NLP,0.91,-1.962974
2,candidate_1.pdf,Python SQL Machine Learning Deep Learning Tens...,0.93,-4.400734
3,candidate_5.pdf,Data Analysis SQL Excel Tableau,0.81,-10.903401
4,candidate_3.pdf,Java Spring Boot Microservices Docker,0.88,-11.106098
